# Five Letters: Kaggle launcher

**Before the first run**
1. Settings (right panel): *Accelerator* **GPU T4 x2**, *Internet* **on**. Both need a verified phone.
2. *Add-ons → Secrets*, add:
   * `GITHUB_TOKEN`: A **GitHub fine-grained token** with read-only *Contents* access to `rl-environments-startup-guide` only (GitHub → Settings → Developer settings → Fine-grained tokens). Save it as the secret `GITHUB_TOKEN`.
   * `HF_TOKEN`: a Hugging Face **write** token.
   * Optional: `GEMINI_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`.
3. Accept the Gemma 3 license on https://huggingface.co/google/gemma-3-270m-it with the same HF account.
4. Keep this notebook **private** (the Kaggle default).

**Running without keeping the tab open:** *Save Version → Save & Run All (Commit)*. It runs headless until done or the session limit. The run stops itself before Kaggle's limit (time budget), saves, and uploads to your HF runs repo.

**Quota:** about 30 GPU hours a week, reset weekly. CPU jobs (traces, solver and API baselines) use no GPU quota: set the accelerator to *None* for them.

In [ ]:
# 1. Settings

REPO = "https://github.com/Aafimalek/rl-environments-startup-guide.git"   # private: needs GITHUB_TOKEN
BRANCH = "development"
SUBDIR = "projects/five-letters"

# What to run. Edit COMMAND for each job. Examples:
#   "sft --config configs/sft_gemma270m.toml --set checkpoint.hub_repo=YOUR_HF_NAME/five-letters-runs --set data.hub_dataset=YOUR_HF_NAME/five-letters-traces"
#   "grpo --config configs/grpo_gemma270m.toml --set checkpoint.hub_repo=YOUR_HF_NAME/five-letters-runs --set model.init=hub:YOUR_HF_NAME/five-letters-runs:sft-gemma270m-v1/best"
#   "eval --policy hf --model hub:YOUR_HF_NAME/five-letters-runs:sft-gemma270m-v1/best --split heldout"
#   "eval --policy hf --model google/gemma-3-270m-it --split heldout"      (zero-shot baseline)
COMMAND = "info"
BASE = "/kaggle/working"

In [ ]:
# 2. Secrets -> environment variables (values are never printed)
import os
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for name in ["GITHUB_TOKEN", "HF_TOKEN", "GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY", "CEREBRAS_API_KEY"]:
        try:
            os.environ[name] = secrets.get_secret(name)
        except Exception:
            pass
except ImportError:
    print("Not running on Kaggle")
print("GITHUB_TOKEN set:", bool(os.environ.get("GITHUB_TOKEN")), "| HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# 3. Get the code (fresh clone, or update an existing one)

import base64, os, subprocess, sys, importlib.metadata as md
CLONE = os.path.join(BASE, "rl-guide")
WORK = os.path.join(CLONE, SUBDIR)

def git(*args):
    """Run git with the token as a header. On failure, show git's message but never the command (it holds the token)."""
    auth = []
    token = os.environ.get("GITHUB_TOKEN", "")
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
    r = subprocess.run(["git", *auth, *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, "***") if token else (r.stderr or r.stdout)
        raise SystemExit(f"git {args[0]} failed: {msg.strip()}\nIs GITHUB_TOKEN set, and does it have read access to the repo?")
    return r.stdout

if os.path.isdir(os.path.join(CLONE, ".git")):
    git("-C", CLONE, "fetch", "--depth", "1", "origin", BRANCH)
    git("-C", CLONE, "reset", "--hard", "FETCH_HEAD")
else:
    git("clone", "--depth", "1", "-b", BRANCH, REPO, CLONE)
print("code at", git("-C", CLONE, "log", "-1", "--oneline").strip())

# The image already has torch. Install our package without touching it,
# and upgrade transformers only if it is too old for Gemma 3.
subprocess.run(f"pip install -q --no-deps -e {WORK}", shell=True, check=True)
def ver(p):
    try:
        return tuple(int(x) for x in md.version(p).split(".")[:2])
    except Exception:
        return (0, 0)
if ver("transformers") < (4, 53) or ver("huggingface_hub") < (0, 25):
    subprocess.run("pip install -q -U 'transformers>=4.53' 'huggingface_hub>=0.25' safetensors", shell=True, check=True)
subprocess.run("pip install -q hf_xet", shell=True)  # faster, deduplicated Hub uploads
os.chdir(WORK)

In [ ]:
# 4. Run

# Exit code 75 means "paused": the run saved a checkpoint (and uploaded it if hub_repo is set).
# Start a new session and run this notebook again to continue from where it stopped.
import subprocess, sys
subprocess.run([sys.executable, "-m", "fiveletters", "info"])
rc = subprocess.run(f"{sys.executable} -m fiveletters {COMMAND}", shell=True).returncode
print({0: "finished", 75: "paused: rerun this notebook to resume"}.get(rc, f"failed with exit code {rc}: read the log above"))